# Sentinel-3 CPR: eRGB, DE2000 Anomaly Detection and CPR Matchup

**Stage 2 of 2.** Run after `Sentinel3_CPR_Pipeline.ipynb`, which downloads and crops the satellite imagery this notebook works from.

## Overview

This notebook carries out the core analysis described in Sections 2.3 to 2.6 of the dissertation. Working from the cropped satellite reflectance boxes produced in Stage 1, it:

1. Converts each pixel to a standardised enhanced RGB (eRGB) colour space (Section 2.3), so satellite colour can be compared against the colour expected from ordinary water constituents.
2. Compares every pixel's colour against two look-up tables (LUTs), one representing ordinary water (Case 2) and one that adds a *C. finmarchicus* absorption component, using the DE2000 colour difference metric, to flag pixels that cannot be explained without astaxanthin pigment (Section 2.4).
3. Converts detected anomalies into a satellite-derived Astaxanthin Equivalent Index (AEI) and matches each satellite box to its corresponding in situ CPR sample (Section 2.5).
4. Statistically compares satellite AEI against in situ AEI for seven taxon combinations, to test how well the two agree and which taxa best explain the satellite signal (Section 2.6).
5. Runs supplementary checks: whether the Calanus LUT genuinely improves the fit over the plain Case 2 model, a restricted analysis of high-abundance points, and the sensitivity of the results to three parameters not fixed in the source literature (matchup time window, minimum valid pixel fraction, DE2000 threshold).

## Settings

Fixed parameters used throughout the notebook, matching the values reported in the Methods (Sections 2.3-2.5):

- **eRGB stretch maxima and gamma** (Section 2.3): fixed reference values from McCarry et al. (2023), not recalculated from the imagery itself, so colour values stay comparable across scenes.
- **DE2000 anomaly threshold** (Section 2.4): 4, following the printing-industry convention that colour differences below this are visually indistinguishable.
- **Satellite AEI conversion factor** (Section 2.5): 0.0201 ug/individual, the mean astaxanthin content across all *C. finmarchicus* life stages. This differs from the 0.05125 ug/individual figure used on the in situ side, because CPR resolves adult stages specifically, whereas the satellite product carries no life-stage information.
- **Matchup quality control** (Section 2.2): a box must have at least 50% valid pixels, and the matched satellite scene must fall within 4 hours of the CPR sample time.

In [ ]:
import os
import re
import glob
import numpy as np
import pandas as pd
import h5py
from datetime import datetime

ROOT = os.path.expanduser('~/CPR_ROI_Sentinel3')

CPR_ROI_FILE    = os.path.join(ROOT, 'CPR_ROI.xlsx')
CPR_FULL_FILE   = os.path.join(ROOT, 'CPR_Data_Original.xlsx')   # optional, for chlorophyll_index
CROPPED_FOLDER  = os.path.join(ROOT, 'Data', 'L2_reprojected', 'L2_cropped')
LUT_C2_PATH     = os.path.join(ROOT, 'Data', 'LUT', 'Eco_Rrs__RGB_Case2.xlsx')
LUT_CAL_PATH    = os.path.join(ROOT, 'Data', 'LUT', 'Eco_Rrs__RGB_Case2_Calanus.xlsx')
OUTPUT_FOLDER   = os.path.join(ROOT, 'Data', 'LUT', 'Output')
STATUS_FOLDER   = os.path.join(ROOT, 'Status')
MATCHUP_FILE    = os.path.join(STATUS_FOLDER, 'cpr_satellite_matchups.csv')

os.makedirs(OUTPUT_FOLDER, exist_ok=True)
os.makedirs(STATUS_FOLDER, exist_ok=True)

# --- McCarry et al. (2023) standardised eRGB stretch ---
# Fixed minimum of 0 and fixed maxima taken as the 90th percentile of each band in the
# Valente et al. (2022) global in situ Rrs dataset, with gamma 0.8 on blue. This is the
# standardised stretch McCarry adopted, NOT the per-image percentile stretch she tested
# and rejected: nothing below is recomputed from the imagery itself.
RED_MAX   = 0.0077   # 555 nm (MODIS) / 560 nm (OLCI Oa06)
GREEN_MAX = 0.0071   # 488 nm (MODIS) / 490 nm (OLCI Oa04)
BLUE_MAX  = 0.0095   # 443 nm (MODIS) / 442.5 nm (OLCI Oa03)
GAMMA     = 0.8

# --- DE2000 anomaly threshold (Shunmugapandi 2025, after the printing-industry
#     convention that differences below 4 are visually insignificant) ---
DE_THRESHOLD = 4.0
DE_THRESHOLDS_TO_TEST = [2.0, 4.0, 6.0, 8.0]

# --- Satellite concentration -> AEI ---
# 0.0201 ug/individual: the CONSTANT mean astaxanthin content, which is what
# Shunmugapandi (2026) applies to the satellite product because it carries no
# life-stage information. (0.05125 is the stage CV/CVI value and belongs only on
# the in situ side, where CPR resolves adult stages.)
CAL_TO_AEI_FACTOR = 0.0201

# --- Matchup QC ---
MIN_VALID_FRACTION = 0.50   # box must have >=50% valid pixels
TIME_WINDOW_HOURS  = 4.0    # primary window, matching Shunmugapandi (2026)
WINDOW_WIDTHS_TO_TEST   = [4, 6, 8, 12, 24]
VALID_FRACTIONS_TO_TEST = [0.50, 0.40, 0.30, 0.20, 0.10]

print(f'Root: {ROOT}')
print(f'Cropped folder: {CROPPED_FOLDER}')
print(f'Satellite AEI factor: {CAL_TO_AEI_FACTOR} ug/ind (constant mean, Shunmugapandi 2026)')

## Load CPR data and build the AEI variants

Loads the CPR reference dataset (`CPR_ROI.xlsx`) and builds the seven in situ AEI variants described in Section 2.5: the individual taxa (*C. finmarchicus*, *C. helgolandicus*, *C. typicus*, Calanus I-IV), and three combined groupings (Calanus adults, all adults, total AEI). These are the values the satellite-derived AEI is compared against later in the notebook.

`roi_row_id` is rebuilt using the same method as in the Stage 1 pipeline notebook, so the IDs stored in each cropped satellite file line up correctly with this table.

In [ ]:
df_roi = pd.read_excel(CPR_ROI_FILE)
df_roi['datetime'] = pd.to_datetime(df_roi['midpoint_date_gmt'], utc=True)
df_roi = df_roi.reset_index(drop=True)
df_roi['roi_row_id'] = df_roi.index
df_roi['month'] = df_roi['datetime'].dt.month
df_roi['year']  = df_roi['datetime'].dt.year

AEI_VARIANTS = {
    'C_fin':                  'Calanus finmarchicus AEI',
    'C_helg':                 'Calanus helgolandicus AEI',
    'C_typicus':              'Centropages typicus AEI',
    'Calanus_I_IV':           'Calanus I-IV AEI',
    'Calanus_adults':         'Calanus adults AEI',
    'All_adults':             'All adults AEI',
    'Total':                  'Total AEI',
}

missing = [c for c in AEI_VARIANTS.values() if c not in df_roi.columns]
if missing:
    print(f'WARNING: expected AEI columns not found: {missing}')
    print(f'Available: {df_roi.columns.tolist()}')
else:
    print(f'All {len(AEI_VARIANTS)} AEI variants available.')

# Optional: pull the CPR chlorophyll index across as a phytoplankton covariate.
# It is the natural control for "is this signal just phytoplankton?".
if os.path.exists(CPR_FULL_FILE):
    try:
        full = pd.read_excel(CPR_FULL_FILE, sheet_name=0,
                             usecols=['midpoint_date_gmt', 'latitude', 'longitude',
                                      'chlorophyll_index'])
        full['datetime'] = pd.to_datetime(full['midpoint_date_gmt'], utc=True)
        df_roi = df_roi.merge(
            full[['datetime', 'latitude', 'longitude', 'chlorophyll_index']],
            on=['datetime', 'latitude', 'longitude'], how='left')
        n_chl = int(df_roi['chlorophyll_index'].notna().sum())
        print(f'chlorophyll_index merged for {n_chl} of {len(df_roi)} points.')
    except Exception as exc:
        print(f'Could not merge chlorophyll_index ({exc}); continuing without it.')
else:
    print('CPR_Data_Original.xlsx not found, continuing without chlorophyll_index.')

print(f'\n{len(df_roi)} CPR points loaded.')
print('\nIn situ AEI summary (ug/m3):')
print(df_roi[list(AEI_VARIANTS.values())].describe(percentiles=[.5, .9, .99]).T[
    ['count', 'mean', '50%', '90%', '99%', 'max']].to_string())
print('\nPresence rates (fraction of points where the taxon was recorded at all):')
for short, col in AEI_VARIANTS.items():
    print(f'  {short:24s} {100 * (df_roi[col] > 0).mean():5.1f}%')

## eRGB conversion (Section 2.3)

Converts each pixel's remote sensing reflectance (Rrs) into the standardised enhanced RGB (eRGB) colour space defined by McCarry et al. (2023): pixels negative in any raw band are excluded, each channel is scaled against a fixed reference maximum, out-of-range pixels are renormalised across their own minimum and maximum, and a gamma correction is applied to the blue channel. This puts every satellite pixel and every LUT entry (below) into the same comparable colour space, which is what makes the DE2000 anomaly comparison meaningful.

In [ ]:
def calc_rgb(red, green, blue):
    """Rrs arrays (560, 490, 442 nm) -> standardised eRGB in [0, 1], NaN where invalid."""
    raw_neg = (red < 0) | (green < 0) | (blue < 0)
    red   = np.where(raw_neg, np.nan, red)
    green = np.where(raw_neg, np.nan, green)
    blue  = np.where(raw_neg, np.nan, blue)

    rgb = np.stack([red / RED_MAX, green / GREEN_MAX, blue / BLUE_MAX], axis=-1).astype(np.float64)
    out = np.full_like(rgb, np.nan)

    valid     = np.all(np.isfinite(rgb), axis=-1)
    over_mask = valid & np.any(rgb > 1, axis=-1)
    ok_mask   = valid & ~over_mask

    if np.any(over_mask):
        pix   = rgb[over_mask]
        pmin  = pix.min(axis=-1, keepdims=True)
        pmax  = pix.max(axis=-1, keepdims=True)
        denom = pmax - pmin
        denom[denom == 0] = 1.0
        out[over_mask] = (pix - pmin) / denom

    out[ok_mask] = rgb[ok_mask]

    b_ch = out[..., 2]
    fin  = np.isfinite(b_ch)
    b_ch[fin] = b_ch[fin] ** GAMMA
    out[..., 2] = b_ch
    return out


print('eRGB conversion function defined.')

## Load the look-up tables, and verify the eRGB conversion

Loads the two bio-optical look-up tables described in Section 2.3: the Case 2 LUT (1,008 combinations of chlorophyll, CDOM, and mineral suspended solids) and the Case 2 + Calanus LUT (1,176 combinations, adding a *C. finmarchicus* concentration axis). Both are read directly from their `RGB_LUT` sheet, i.e. McCarry's own finished R, G, B values, rather than recalculated from scratch.

This cell also runs a validation check: it reproduces each LUT's RGB values from its own underlying Rrs spectra using the `calc_rgb` function above, and compares the two. Close agreement is evidence the eRGB conversion has been implemented correctly.

In [ ]:
import subprocess, sys
try:
    from skimage.color import rgb2lab, deltaE_ciede2000
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'scikit-image', '-q',
                           '--break-system-packages'])
    from skimage.color import rgb2lab, deltaE_ciede2000


def load_lut(lut_path):
    """Read McCarry's precomputed eRGB LUT. Returns (lut_df, conc_columns)."""
    lut = pd.read_excel(lut_path, sheet_name='RGB_LUT', header=0)
    lut.columns = [str(c).strip() for c in lut.columns]
    for c in ('R', 'G', 'B'):
        if c not in lut.columns:
            raise ValueError(f'{lut_path}: RGB_LUT sheet has no "{c}" column. '
                             f'Columns found: {lut.columns.tolist()}')
    conc_cols = [c for c in lut.columns if c not in ('R', 'G', 'B')]
    lut = lut.dropna(subset=['R', 'G', 'B']).reset_index(drop=True)
    for c in ['R', 'G', 'B'] + conc_cols:
        lut[c] = pd.to_numeric(lut[c], errors='coerce')
    return lut.dropna(subset=['R', 'G', 'B']).reset_index(drop=True), conc_cols


def verify_lut_against_rrs(lut_path, lut_df, wl_rgb=(554, 488, 442)):
    """Reproduce the supplied RGB_LUT from the Rrs sheet using calc_rgb, as a check that
    the port of calcRGB.m is faithful. Returns (median_abs_err, max_abs_err, n_compared)."""
    rrs = pd.read_excel(lut_path, sheet_name='Rrs', header=None)
    header = rrs.iloc[1]
    wl_col = {}
    for i, v in enumerate(header):
        try:
            wl_col[float(v)] = i
        except (TypeError, ValueError):
            pass
    data = rrs.iloc[2:].reset_index(drop=True)
    r_w, g_w, b_w = wl_rgb
    r = data.iloc[:, wl_col[r_w]].astype(float).values
    g = data.iloc[:, wl_col[g_w]].astype(float).values
    b = data.iloc[:, wl_col[b_w]].astype(float).values
    rebuilt = calc_rgb(r, g, b)
    n = min(len(rebuilt), len(lut_df))
    err = np.abs(rebuilt[:n] - lut_df[['R', 'G', 'B']].values[:n])
    return float(np.nanmedian(err)), float(np.nanmax(err)), n


print('Loading LUTs from their RGB_LUT sheets (McCarry\'s own values, not recomputed)...')
lut_c2,  conc_c2  = load_lut(LUT_C2_PATH)
lut_cal, conc_cal = load_lut(LUT_CAL_PATH)
print(f'  Case 2:           {len(lut_c2):5d} entries, constituents: {conc_c2}')
print(f'  Case 2 + Calanus: {len(lut_cal):5d} entries, constituents: {conc_cal}')

cal_candidates = [c for c in conc_cal if 'cal' in c.lower()]
if not cal_candidates:
    raise ValueError(f'No Calanus concentration column found. Columns: {conc_cal}')
CAL_CONC_COL = cal_candidates[0]
print(f'  Calanus column: "{CAL_CONC_COL}", '
      f'rungs: {sorted(lut_cal[CAL_CONC_COL].unique())}')

# --- MSS coverage check: flags whether this is the original or updated Calanus LUT ---
mss_c2  = sorted(lut_c2[[c for c in conc_c2 if 'MSS' in c][0]].unique())
mss_cal = sorted(lut_cal[[c for c in conc_cal if 'MSS' in c][0]].unique())
print(f'\n  Case 2 MSS values:           {mss_c2}')
print(f'  Case 2 + Calanus MSS values: {mss_cal}')
if len(mss_cal) == 1:
    print('  NOTE: the Calanus LUT holds MSS fixed. This is the ORIGINAL McCarry LUT, not the')
    print('        updated one in Shunmugapandi (2025) where MSS is varied. In turbid water the')
    print('        Calanus LUT has no sediment freedom, so it can fit worse than Case 2 and can')
    print('        absorb sediment as spurious Calanus. Report this as a limitation.')

# --- validation of the calcRGB.m port ---
print('\nVerifying the eRGB port by reproducing each RGB_LUT sheet from its own Rrs sheet:')
for name, path, df in [('Case 2', LUT_C2_PATH, lut_c2),
                       ('Case 2 + Calanus', LUT_CAL_PATH, lut_cal)]:
    try:
        med, mx, n = verify_lut_against_rrs(path, df)
        verdict = 'PASS' if med < 0.01 else 'CHECK THIS'
        print(f'  {name:18s} n={n:5d}  median |error| {med:.5f}  max {mx:.5f}   {verdict}')
    except Exception as exc:
        print(f'  {name:18s} verification could not run: {exc}')

lut_c2_lab  = rgb2lab(lut_c2[['R', 'G', 'B']].values.astype(np.float64).reshape(1, -1, 3)).reshape(-1, 3)
lut_cal_lab = rgb2lab(lut_cal[['R', 'G', 'B']].values.astype(np.float64).reshape(1, -1, 3)).reshape(-1, 3)
CAL_CONC_VALUES = lut_cal[CAL_CONC_COL].values.astype(np.float64)
print('\nLUTs converted to L*a*b.')


def compute_de2000(pixel_rgb_flat, lut_lab, chunk=4000):
    """Per pixel, the minimum DE2000 against the LUT and the index of that best match."""
    n = pixel_rgb_flat.shape[0]
    de_min   = np.full(n, np.nan, dtype=np.float64)
    best_idx = np.zeros(n, dtype=np.int32)
    for start in range(0, n, chunk):
        end       = min(start + chunk, n)
        chunk_lab = rgb2lab(pixel_rgb_flat[start:end].reshape(1, -1, 3)).reshape(-1, 3)
        de        = deltaE_ciede2000(chunk_lab[:, None, :], lut_lab[None, :, :])
        idx       = np.argmin(de, axis=1)
        de_min[start:end]   = de[np.arange(end - start), idx]
        best_idx[start:end] = idx
    return de_min, best_idx


print('DE2000 matching functions defined.')

## Quality control on the saved spectra (Section 2.2)

Defines the pixel-level quality check applied before the eRGB conversion: **negative Rrs at any wavelength**, the criterion used by McCarry (2023) and Shunmugapandi (2025) to flag likely atmospheric correction failures. This is possible here because Stage 1 saves all twelve OLCI bands rather than only the three used for eRGB.

In [ ]:
# Wavelengths of the saved bands (OLCI Oa01-Oa12)
BAND_WAVELENGTHS = {
    'Rrs_400': 400.00, 'Rrs_412': 412.50, 'Rrs_442': 442.50, 'Rrs_490': 490.00,
    'Rrs_510': 510.00, 'Rrs_560': 560.00, 'Rrs_620': 620.00, 'Rrs_665': 665.00,
    'Rrs_674': 673.75, 'Rrs_681': 681.25, 'Rrs_709': 708.75, 'Rrs_754': 753.75,
}
def negative_spectrum_mask(band_dict):
    """True where the pixel is negative at ANY available wavelength (McCarry / Shunmugapandi QC)."""
    stack = np.stack([band_dict[b] for b in sorted(band_dict)], axis=0)
    with np.errstate(invalid='ignore'):
        return np.any(stack < 0, axis=0)


print('QC functions defined.')
print(f'  negative-Rrs check: across all saved bands')

## Scene timestamp parsing

Extracts the satellite acquisition time from each Sentinel-3 scene's filename, used to calculate how many hours separate the satellite observation from the corresponding CPR sample time (Section 2.2). This time difference is the basis for the 4-hour matchup window applied later in the notebook.

In [ ]:
def parse_scene_datetime(scene_name):
    """Midpoint UTC datetime of a Sentinel-3 scene, from its name."""
    matches = re.findall(r'(\d{8}T\d{6})', scene_name)
    if len(matches) < 2:
        return None
    t1 = datetime.strptime(matches[0], '%Y%m%dT%H%M%S')
    t2 = datetime.strptime(matches[1], '%Y%m%dT%H%M%S')
    return (t1 + (t2 - t1) / 2).replace(tzinfo=None)


print('Timestamp parser defined.')

## Anomaly detection and AEI calculation for every box (Sections 2.4-2.5)

The central analysis step. For every cropped satellite box:

1. Applies the quality control checks above.
2. Converts valid pixels to eRGB.
3. Compares each pixel against both LUTs using DE2000, and calls a pixel anomalous if its Case 2 DEmin meets or exceeds the threshold.
4. Converts anomalous pixels' best-matching Case 2 + Calanus LUT concentration into a satellite AEI value.
5. Reduces each box to summary statistics (mean DE2000, percentage of anomalous pixels, mean AEI) used in the matchup and statistical analysis that follows.

Statistics are computed from the raw satellite swath pixels rather than the resampled display grid, so no resampling artefacts enter the results (Section 2.2).

In [ ]:
cropped_files = sorted(glob.glob(os.path.join(CROPPED_FOLDER, '*_cropped.h5')))
print(f'Found {len(cropped_files)} cropped files\n')
if not cropped_files:
    raise FileNotFoundError(f'No cropped files in {CROPPED_FOLDER}. Run the pipeline notebook first.')

roi_datetime_lookup = df_roi.set_index('roi_row_id')['datetime'].to_dict()

matchup_records = []
failed_files    = []

for n_done, filepath in enumerate(cropped_files, 1):
    base = os.path.basename(filepath).replace('_cropped.h5', '')
    try:
        with h5py.File(filepath, 'r') as f:
            roi_row_id = int(f.attrs.get('roi_row_id', -1))
            roi_lat    = float(f.attrs.get('roi_lat', np.nan))
            roi_lon    = float(f.attrs.get('roi_lon', np.nan))
            scene_name = str(f.attrs.get('scene_name', ''))
            scene_date = str(f.attrs.get('scene_date', ''))

            if 'swath' in f:
                src   = f['swath']
                bands = {k: src[k][:].astype(np.float64)
                         for k in src.keys() if k.startswith('Rrs')}
                n_box_pixels = int(src['latitude'].shape[0])
            else:
                # backwards compatibility with v1 crops (grid only, three bands)
                bands = {k: f[k][:].astype(np.float64).ravel()
                         for k in f.keys() if k.startswith('Rrs')}
                n_box_pixels = int(next(iter(bands.values())).size)

        rec = {
            'roi_row_id': roi_row_id, 'scene_name': scene_name, 'scene_date': scene_date,
            'filepath': filepath, 'roi_lat': roi_lat, 'roi_lon': roi_lon,
            'n_box_pixels': n_box_pixels,
        }

        # --- time difference (filtering happens downstream) ---
        scene_dt = parse_scene_datetime(scene_name)
        cpr_dt   = roi_datetime_lookup.get(roi_row_id)
        if scene_dt is None or cpr_dt is None:
            rec.update(status='no_timestamp', reason='could not parse scene or CPR time',
                       time_diff_hours=np.nan)
            matchup_records.append(rec)
            continue
        rec['time_diff_hours'] = abs(
            (scene_dt - pd.Timestamp(cpr_dt).tz_localize(None)).total_seconds()) / 3600.0

        if not all(b in bands for b in ('Rrs_442', 'Rrs_490', 'Rrs_560')):
            rec.update(status='missing_ergb_bands', reason=f'bands present: {sorted(bands)}')
            matchup_records.append(rec)
            continue

        # ---------------- quality control ----------------
        neg_mask = negative_spectrum_mask(bands)
        rec['n_negative_spectra'] = int(np.nansum(neg_mask))

        drop = neg_mask.copy()
        for b in bands:
            bands[b] = np.where(drop, np.nan, bands[b])

        # ---------------- eRGB ----------------
        rgb = calc_rgb(bands['Rrs_560'], bands['Rrs_490'], bands['Rrs_442'])
        valid_mask = np.all(np.isfinite(rgb), axis=-1)
        n_valid = int(valid_mask.sum())
        rec['n_valid_pixels']  = n_valid
        rec['valid_fraction']  = n_valid / n_box_pixels if n_box_pixels else 0.0

        if n_valid == 0:
            rec.update(status='no_valid_pixels', reason='zero valid pixels after QC')
            matchup_records.append(rec)
            continue

        # turbidity / phytoplankton proxies from the box itself, used later as controls
        rec['mean_Rrs_560'] = float(np.nanmean(bands['Rrs_560'][valid_mask]))
        if 'Rrs_665' in bands:
            rec['mean_Rrs_665'] = float(np.nanmean(bands['Rrs_665'][valid_mask]))

        # ---------------- DE2000 against BOTH LUTs ----------------
        pix = rgb[valid_mask].astype(np.float64)
        de_c2,  _        = compute_de2000(pix, lut_c2_lab)
        de_cal, idx_cal  = compute_de2000(pix, lut_cal_lab)
        cal_conc_all     = CAL_CONC_VALUES[idx_cal]

        rec['median_DE_C2']  = float(np.median(de_c2))
        rec['median_DE_Cal'] = float(np.median(de_cal))

        # primary threshold
        anom = de_c2 >= DE_THRESHOLD
        rec['n_anomalous_pixels'] = int(anom.sum())
        rec['pct_anomalous']      = 100.0 * anom.mean()

        if anom.any():
            rec['median_DE_C2_anom']  = float(np.median(de_c2[anom]))
            rec['median_DE_Cal_anom'] = float(np.median(de_cal[anom]))
            rec['median_DE_reduction'] = float(np.median(de_c2[anom] - de_cal[anom]))
            rec['frac_anom_improved']  = float(np.mean(de_cal[anom] < de_c2[anom]))
        else:
            rec['median_DE_C2_anom'] = rec['median_DE_Cal_anom'] = np.nan
            rec['median_DE_reduction'] = rec['frac_anom_improved'] = np.nan

        # DE reduction over ALL valid pixels, for the paired test (not threshold dependent)
        rec['median_DE_reduction_all'] = float(np.median(de_c2 - de_cal))

        # box concentration at every threshold, so the threshold is free downstream
        for T in DE_THRESHOLDS_TO_TEST:
            a = de_c2 >= T
            conc = np.where(a, cal_conc_all, 0.0)
            rec[f'satellite_cal_mean_T{T:g}'] = float(conc.mean())
            rec[f'pct_anomalous_T{T:g}']      = 100.0 * a.mean()

        conc_primary = np.where(anom, cal_conc_all, 0.0)
        rec['satellite_cal_mean']     = float(conc_primary.mean())
        rec['satellite_cal_mean_pos'] = (float(conc_primary[conc_primary > 0].mean())
                                         if np.any(conc_primary > 0) else np.nan)
        rec['satellite_aei']          = rec['satellite_cal_mean'] * CAL_TO_AEI_FACTOR
        rec['satellite_aei_pos']      = rec['satellite_cal_mean_pos'] * CAL_TO_AEI_FACTOR
        rec['n_pixels_cal_positive']  = int((conc_primary > 0).sum())

        rec['status'] = 'ok'
        rec['reason'] = (f'{rec["valid_fraction"]:.0%} valid, '
                         f'{rec["n_anomalous_pixels"]}/{n_valid} anomalous, '
                         f'{rec["time_diff_hours"]:.1f}h from CPR time')
        matchup_records.append(rec)

        if n_done % 50 == 0 or n_done == len(cropped_files):
            print(f'  {n_done}/{len(cropped_files)} files processed...')

    except Exception as exc:
        print(f'FAILED: {base}  ({type(exc).__name__}: {exc})')
        failed_files.append((base, str(exc)))

matchup_df = pd.DataFrame(matchup_records)
print(f'\nProcessed {len(cropped_files)} files, {len(failed_files)} failed.')
for name, err in failed_files:
    print(f'  {name}: {err}')

print('\nStatus breakdown:')
print(matchup_df['status'].value_counts().to_string())

ok = matchup_df[matchup_df['status'] == 'ok']
if len(ok):
    print(f'\nQC effect across {len(ok)} usable boxes:')
    print(f'  negative-Rrs pixels removed: {int(ok["n_negative_spectra"].sum()):,}')
    print(f'  median valid fraction: {ok["valid_fraction"].median():.1%}')
    print(f'  median anomalous pixel fraction at DE>={DE_THRESHOLD}: '
          f'{ok["pct_anomalous"].median():.1f}%')

## Select primary matchups and join to the in situ AEI (Section 2.2)

Applies the matchup rule described in the Methods: for each CPR point, the closest satellite scene within the time window that also meets the minimum valid pixel fraction is kept as the "primary matchup," then joined to that point's in situ AEI values. This is the step that produces the 97 primary matchups reported in Results Section 3.1 (Table 2).

In [ ]:
aei_cols = ['roi_row_id', 'month', 'year'] + list(AEI_VARIANTS.values())
if 'chlorophyll_index' in df_roi.columns:
    aei_cols.append('chlorophyll_index')


def select_primary_matchups(mdf, time_window_hours, min_valid_fraction=MIN_VALID_FRACTION,
                            de_threshold=DE_THRESHOLD):
    """Closest-in-time qualifying scene per CPR point, joined to the in situ AEI variants."""
    cand = mdf[(mdf['status'] == 'ok') &
               (mdf['time_diff_hours'] <= time_window_hours) &
               (mdf['valid_fraction'] >= min_valid_fraction)].copy()
    if len(cand) == 0:
        return pd.DataFrame(columns=['roi_row_id', 'satellite_aei'] + list(AEI_VARIANTS.values()))

    best = cand.loc[cand.groupby('roi_row_id')['time_diff_hours'].idxmin()].copy()

    # allow a non-default DE threshold without re-running the DE2000 step
    col = f'satellite_cal_mean_T{de_threshold:g}'
    if col in best.columns:
        best['satellite_cal_mean'] = best[col]
        best['satellite_aei']      = best[col] * CAL_TO_AEI_FACTOR
        best['pct_anomalous']      = best[f'pct_anomalous_T{de_threshold:g}']

    return best.merge(df_roi[aei_cols], on='roi_row_id', how='left')


merged = select_primary_matchups(matchup_df, TIME_WINDOW_HOURS, MIN_VALID_FRACTION)
merged.to_csv(MATCHUP_FILE, index=False)
matchup_df.to_csv(MATCHUP_FILE.replace('.csv', '_all_candidates.csv'), index=False)

print(f'{len(df_roi)} CPR points total')
print(f'{matchup_df["roi_row_id"].nunique()} have at least one cropped box')
print(f'{len(merged)} have a primary matchup at +/-{TIME_WINDOW_HOURS}h '
      f'and >={MIN_VALID_FRACTION:.0%} valid pixels')
if len(merged):
    print(f'  satellite anomaly detected in {(merged["satellite_aei"] > 0).sum()} of them '
          f'({100 * (merged["satellite_aei"] > 0).mean():.0f}%)')
    print(f'  median time offset: {merged["time_diff_hours"].median():.1f}h')
print(f'\nSaved: {MATCHUP_FILE}')

## Does adding Calanus to the model actually improve the fit? (Section 2.4, Results 3.2)

Tests whether the Case 2 + Calanus LUT provides a genuinely better fit than the plain Case 2 LUT, rather than the improvement being an artefact of how the anomaly threshold is defined. Uses a paired Wilcoxon signed-rank test on the change in DEmin, with two additional checks:

- whether the improvement is larger at points where CPR actually recorded Calanus (Mann-Whitney U test), and
- whether the improvement instead tracks a turbidity proxy (Spearman correlation), which would suggest the fixed-sediment Calanus LUT is absorbing a sediment signal rather than reflecting Calanus.

In [ ]:
from scipy import stats as scipy_stats

sub = merged.dropna(subset=['median_DE_reduction']) if len(merged) else merged

print('=== Does adding Calanus to the bio-optical model reduce the anomaly? ===\n')
if len(sub) < 6:
    print(f'Only {len(sub)} matched point(s) have anomalous pixels; not enough to test.')
else:
    red = sub['median_DE_reduction'].values
    stat, p = scipy_stats.wilcoxon(red, alternative='two-sided')
    print(f'All matched points with anomalous pixels (n = {len(sub)}):')
    print(f'  median DE reduction   {np.median(red):+.3f}')
    print(f'  points improved       {int((red > 0).sum())} / {len(red)} '
          f'({100 * (red > 0).mean():.0f}%)')
    print(f'  Wilcoxon signed-rank  W = {stat:.1f}, p = {p:.4g}')
    print(f'  -> {"reduction is significant" if p < 0.05 else "no significant reduction"}')

    # Control 1: points with no Calanus recorded in situ
    zero = sub[sub['Calanus adults AEI'] == 0]['median_DE_reduction'].dropna().values
    pos  = sub[sub['Calanus adults AEI'] > 0]['median_DE_reduction'].dropna().values
    print(f'\nControl 1, split by whether CPR recorded any Calanus adults:')
    print(f'  no Calanus recorded   n = {len(zero):4d}, median reduction {np.median(zero):+.3f}'
          if len(zero) else '  no Calanus recorded   n = 0')
    print(f'  Calanus recorded      n = {len(pos):4d}, median reduction {np.median(pos):+.3f}'
          if len(pos) else '  Calanus recorded      n = 0')
    if len(zero) >= 5 and len(pos) >= 5:
        u, pu = scipy_stats.mannwhitneyu(pos, zero, alternative='two-sided')
        print(f'  Mann-Whitney U = {u:.1f}, p = {pu:.4g}')
        print(f'  -> {"reduction differs with Calanus presence" if pu < 0.05 else "no difference; the reduction is not tracking Calanus presence"}')

    # Control 2: turbidity
    if 'mean_Rrs_560' in sub.columns and sub['mean_Rrs_560'].notna().sum() >= 6:
        t = sub.dropna(subset=['mean_Rrs_560', 'median_DE_reduction'])
        rho, pr = scipy_stats.spearmanr(t['mean_Rrs_560'], t['median_DE_reduction'])
        print(f'\nControl 2, DE reduction against a turbidity proxy (mean Rrs 560 nm), n = {len(t)}:')
        print(f'  Spearman rho = {rho:+.3f}, p = {pr:.4g}')
        print('  -> a strong positive rho would suggest the Calanus LUT is absorbing sediment,')
        print('     which is a live risk here because its MSS is fixed at 0.03 g/m3.')

## Correlation statistics per AEI variant (Section 2.6, Results 3.3)

Compares satellite-derived AEI against each of the seven in situ AEI variants. Because both sides are heavily zero-inflated (*C. finmarchicus* is absent from most CPR samples, and most boxes have no satellite detection), the comparison is split into two parts:

- **Detection**: whether an anomaly is more likely to be flagged where in situ abundance is higher (Mann-Whitney U test, 2x2 presence table with Fisher's exact test, Cohen's kappa).
- **Magnitude**: among points positive on both sides, the log-log Pearson correlation reported by Shunmugapandi (2026), plus a Spearman correlation across all matched points (including zeros) as the primary agreement statistic given the zero inflation.

In [ ]:
def compute_stats(mdf, verbose=True, label=''):
    """Detection and magnitude statistics for every AEI variant."""
    rows = []
    for short, col in AEI_VARIANTS.items():
        if 'satellite_aei' not in mdf.columns or col not in mdf.columns:
            rows.append({'variant': short, 'n': 0})
            continue

        d = mdf[['satellite_aei', col]].dropna().rename(columns={col: 'insitu'})
        n = len(d)
        r = {'variant': short, 'n': n}
        if n == 0:
            rows.append(r)
            continue

        det = d['satellite_aei'] > 0
        pres = d['insitu'] > 0
        r['n_detected'] = int(det.sum())
        r['n_present']  = int(pres.sum())

        # --- detection half ---
        a, b = d.loc[det, 'insitu'].values, d.loc[~det, 'insitu'].values
        if len(a) >= 3 and len(b) >= 3:
            u, pu = scipy_stats.mannwhitneyu(a, b, alternative='two-sided')
            r['detect_U'] = u
            r['detect_p'] = pu
            r['detect_AUC'] = u / (len(a) * len(b))     # P(detected point has higher in situ AEI)
            r['detect_median_insitu_yes'] = float(np.median(a))
            r['detect_median_insitu_no']  = float(np.median(b))
        else:
            for k in ('detect_U', 'detect_p', 'detect_AUC',
                      'detect_median_insitu_yes', 'detect_median_insitu_no'):
                r[k] = np.nan

        # --- 2x2 presence table ---
        tab = np.array([[int((~det & ~pres).sum()), int((~det & pres).sum())],
                        [int(( det & ~pres).sum()), int(( det & pres).sum())]])
        r['tab_nn'], r['tab_np'], r['tab_pn'], r['tab_pp'] = tab.ravel()
        if tab.sum() > 0 and tab.min() >= 0:
            try:
                _, r['fisher_p'] = scipy_stats.fisher_exact(tab)
            except Exception:
                r['fisher_p'] = np.nan
            po = np.trace(tab) / tab.sum()
            pe = ((tab.sum(0) * tab.sum(1)).sum()) / tab.sum() ** 2
            r['kappa'] = (po - pe) / (1 - pe) if pe < 1 else np.nan
        else:
            r['fisher_p'] = r['kappa'] = np.nan

        # --- magnitude half ---
        bp = d[(d['satellite_aei'] > 0) & (d['insitu'] > 0)]
        r['n_both_positive'] = len(bp)
        if len(bp) >= 3:
            ls, li = np.log10(bp['satellite_aei']), np.log10(bp['insitu'])
            pr_, pp_ = scipy_stats.pearsonr(ls, li)
            sr_, sp_ = scipy_stats.spearmanr(bp['satellite_aei'], bp['insitu'])
            r.update(pearson_r_loglog=pr_, r_squared=pr_ ** 2, pearson_p=pp_,
                     spearman_r=sr_, spearman_p=sp_,
                     median_ratio=float(np.median(bp['satellite_aei'] / bp['insitu'])))
        else:
            for k in ('pearson_r_loglog', 'r_squared', 'pearson_p',
                      'spearman_r', 'spearman_p', 'median_ratio'):
                r[k] = np.nan

        # --- rank correlation on ALL matched points, zeros included ---
        # Keeps the full sample; the appropriate headline number when both sides are
        # zero-inflated, since it does not condition on a positive detection.
        if n >= 5:
            sr_all, sp_all = scipy_stats.spearmanr(d['satellite_aei'], d['insitu'])
            r['spearman_r_all'] = sr_all
            r['spearman_p_all'] = sp_all
        else:
            r['spearman_r_all'] = r['spearman_p_all'] = np.nan

        rows.append(r)

    res = pd.DataFrame(rows)
    if verbose and len(res):
        print(f'=== Correlation statistics {label} ===\n')
        show = ['variant', 'n', 'n_detected', 'n_present', 'detect_AUC', 'detect_p',
                'kappa', 'fisher_p', 'spearman_r_all', 'spearman_p_all',
                'n_both_positive', 'pearson_r_loglog', 'r_squared', 'pearson_p']
        show = [c for c in show if c in res.columns]
        print(res[show].to_string(index=False,
              float_format=lambda v: f'{v:.3g}' if pd.notna(v) else 'nan'))
    return res


stats_df = compute_stats(merged, label=f'(+/-{TIME_WINDOW_HOURS}h, '
                                       f'>={MIN_VALID_FRACTION:.0%} valid)')
STATS_FILE = os.path.join(STATUS_FOLDER, 'cpr_satellite_stats_summary.csv')
stats_df.to_csv(STATS_FILE, index=False)
print(f'\nSaved: {STATS_FILE}')
print('\nHow to read this:')
print('  detect_AUC  probability a point with a satellite detection has higher in situ AEI')
print('              than one without. 0.5 = no relationship, >0.5 = detection tracks abundance.')
print('  spearman_r_all  rank correlation over ALL matched points including zeros. This is the')
print('              headline number when both sides are zero-inflated.')
print('  pearson_r_loglog  Shunmugapandi\'s statistic, on the both-positive subset only.')

## Which taxon or combination best explains the signal? (Section 2.6)

Ranks the seven in situ AEI variants by how strongly each correlates with satellite AEI, then uses Williams' test to check whether the best-performing variant is genuinely a better explanation than the others, rather than differing from them only by chance (the variants are not independent, since they all share the same satellite variable). Partial correlations are also run, controlling for chlorophyll and turbidity, to check whether any apparent taxon-satellite relationship might actually be a confound.

In [ ]:
def williams_test(r_xy, r_xz, r_yz, n):
    """Williams' t for two DEPENDENT correlations sharing variable x. Returns (t, p, df)."""
    if not all(np.isfinite([r_xy, r_xz, r_yz])) or n < 5:
        return np.nan, np.nan, np.nan
    det = 1 - r_xy ** 2 - r_xz ** 2 - r_yz ** 2 + 2 * r_xy * r_xz * r_yz
    if det <= 0:
        return np.nan, np.nan, np.nan
    num = (r_xy - r_xz) * np.sqrt((n - 1) * (1 + r_yz))
    den = np.sqrt(2 * ((n - 1) / (n - 3)) * det + ((r_xy + r_xz) ** 2 / 4) * (1 - r_yz) ** 3)
    if den == 0:
        return np.nan, np.nan, np.nan
    t = num / den
    df = n - 3
    return t, 2 * (1 - scipy_stats.t.cdf(abs(t), df)), df


def partial_spearman(x, y, z):
    """Spearman correlation of x and y controlling for z, via ranks and residuals."""
    m = np.isfinite(x) & np.isfinite(y) & np.isfinite(z)
    if m.sum() < 6:
        return np.nan, np.nan, int(m.sum())
    rx, ry, rz = (scipy_stats.rankdata(v[m]) for v in (x, y, z))
    ex = rx - np.polyval(np.polyfit(rz, rx, 1), rz)
    ey = ry - np.polyval(np.polyfit(rz, ry, 1), rz)
    r, _ = scipy_stats.pearsonr(ex, ey)
    n = int(m.sum())
    if n <= 4:
        return r, np.nan, n
    t = r * np.sqrt((n - 3) / max(1 - r ** 2, 1e-12))
    return r, 2 * (1 - scipy_stats.t.cdf(abs(t), n - 3)), n


print('=== Ranking the AEI variants by how well they explain the satellite signal ===\n')
if len(merged) < 8:
    print(f'Only {len(merged)} matched point(s); not enough for a taxon comparison.')
else:
    sat = merged['satellite_aei'].values
    corrs, ns = {}, {}
    for short, col in AEI_VARIANTS.items():
        v = merged[col].values
        m = np.isfinite(sat) & np.isfinite(v)
        if m.sum() >= 6:
            corrs[short] = scipy_stats.spearmanr(sat[m], v[m])[0]
            ns[short] = int(m.sum())

    ranked = sorted(corrs.items(), key=lambda kv: -abs(kv[1]))
    print('Spearman correlation with satellite AEI (all matched points, zeros included):')
    for k, v in ranked:
        print(f'  {k:24s} rho = {v:+.3f}   (n = {ns[k]})')

    best = ranked[0][0]
    print(f'\nBest single explanator: {best} (rho = {corrs[best]:+.3f})')
    print(f'\nWilliams tests, {best} against each other variant '
          f'(is it genuinely better, or just noise?):')
    for other, r_other in ranked[1:]:
        m = (np.isfinite(sat) & np.isfinite(merged[AEI_VARIANTS[best]].values)
             & np.isfinite(merged[AEI_VARIANTS[other]].values))
        n = int(m.sum())
        r_yz = scipy_stats.spearmanr(merged[AEI_VARIANTS[best]].values[m],
                                     merged[AEI_VARIANTS[other]].values[m])[0]
        t, p, df = williams_test(corrs[best], r_other, r_yz, n)
        verdict = ('significantly better' if (np.isfinite(p) and p < 0.05)
                   else 'not distinguishable')
        print(f'  vs {other:24s} t = {t:+.2f}, df = {df}, p = {p:.3g}   {verdict}'
              if np.isfinite(t) else f'  vs {other:24s} could not be computed')

    # --- confounder controls ---
    print('\n=== Partial correlations: is the signal really the zooplankton? ===')
    for ctrl_name, ctrl_col in [('CPR chlorophyll index', 'chlorophyll_index'),
                                ('turbidity proxy (Rrs 560)', 'mean_Rrs_560')]:
        if ctrl_col not in merged.columns or merged[ctrl_col].notna().sum() < 8:
            print(f'\n  {ctrl_name}: not available, skipped.')
            continue
        print(f'\n  Controlling for {ctrl_name}:')
        for short in [k for k, _ in ranked[:4]]:
            raw = corrs[short]
            pr, pp, n = partial_spearman(sat, merged[AEI_VARIANTS[short]].values,
                                         merged[ctrl_col].values)
            if np.isfinite(pr):
                drop = abs(raw) - abs(pr)
                note = 'largely explained by the control' if drop > abs(raw) * 0.5 else 'survives'
                print(f'    {short:24s} rho {raw:+.3f} -> partial {pr:+.3f} '
                      f'(p = {pp:.3g}, n = {n})  {note}')

## Targeted subset: high-abundance points only

A supplementary check, following McCarry's advice that the method is not intended to be applied blanket-style and is expected to perform best where zooplankton surface concentrations are genuinely high. This restricts the matchups to the top decile of a chosen AEI variant and re-runs the detection rate and correlation statistics within that subset, to see whether the method performs better under the conditions it was designed for. This is exploratory and supplementary to the main blanket analysis reported in the Results.

In [ ]:
TARGET_VARIANT   = 'Calanus_adults'   # the astaxanthin-bearing Calanus signal
TARGET_PERCENTILE = 90                   # top decile

print(f'=== Targeted subset: top {100 - TARGET_PERCENTILE}% of points by '
      f'{TARGET_VARIANT} ===\n')

if len(merged) < 10:
    print(f'Only {len(merged)} matched point(s); the targeted subset is not meaningful yet.')
else:
    tcol = AEI_VARIANTS[TARGET_VARIANT]
    cut = np.nanpercentile(df_roi[tcol], TARGET_PERCENTILE)
    high = merged[merged[tcol] >= cut]
    low  = merged[merged[tcol] <  cut]
    print(f'Threshold: {tcol} >= {cut:.4f} ug/m3')
    print(f'  high-abundance matchups: {len(high)}')
    print(f'  rest:                    {len(low)}')

    if len(high) >= 5 and len(low) >= 5:
        dh = 100 * (high['satellite_aei'] > 0).mean()
        dl = 100 * (low['satellite_aei'] > 0).mean()
        print(f'\nSatellite detection rate:')
        print(f'  high-abundance points {dh:.0f}%')
        print(f'  other points          {dl:.0f}%   <- an estimate of the false positive rate')
        tab = np.array([
            [int((low['satellite_aei'] <= 0).sum()),  int((low['satellite_aei'] > 0).sum())],
            [int((high['satellite_aei'] <= 0).sum()), int((high['satellite_aei'] > 0).sum())]])
        try:
            odds, pf = scipy_stats.fisher_exact(tab)
            print(f'  Fisher exact: odds ratio {odds:.2f}, p = {pf:.4g}')
            print(f'  -> {"detection is associated with high abundance" if pf < 0.05 else "detection is NOT associated with high abundance"}')
        except Exception as exc:
            print(f'  Fisher exact could not be computed: {exc}')

        if len(high) >= 8:
            print(f'\nCorrelations within the targeted subset only (n = {len(high)}):')
            stats_high = compute_stats(high, verbose=False)
            cols = ['variant', 'n', 'n_both_positive', 'spearman_r_all',
                    'spearman_p_all', 'pearson_r_loglog', 'pearson_p']
            print(stats_high[[c for c in cols if c in stats_high.columns]].to_string(
                index=False, float_format=lambda v: f'{v:.3g}' if pd.notna(v) else 'nan'))
            stats_high.to_csv(os.path.join(STATUS_FOLDER,
                              'cpr_satellite_stats_targeted.csv'), index=False)
    else:
        print('\nToo few points in one of the groups to compare.')

## Sensitivity checks

Tests how sensitive the results are to three parameters that are not fixed by the source literature: the matchup time window, the minimum valid pixel fraction, and the DE2000 anomaly threshold. Included to show whether the main findings are robust to these choices, or whether they depend heavily on one particular setting.

In [ ]:
def sweep(param_name, values, build):
    rows = []
    for v in values:
        m = build(v)
        s = compute_stats(m, verbose=False)
        tot = s[s['variant'] == 'Total']
        cfin = s[s['variant'] == 'C_fin']
        rows.append({
            param_name: v,
            'n_matched': len(m),
            'n_detected': int((m['satellite_aei'] > 0).sum()) if len(m) else 0,
            'Total_spearman_all': tot['spearman_r_all'].iloc[0] if len(tot) else np.nan,
            'Total_spearman_p':   tot['spearman_p_all'].iloc[0] if len(tot) else np.nan,
            'C_fin_spearman_all': cfin['spearman_r_all'].iloc[0] if len(cfin) else np.nan,
            'Total_n_both_pos':   tot['n_both_positive'].iloc[0] if len(tot) else np.nan,
            'Total_pearson_loglog': tot['pearson_r_loglog'].iloc[0] if len(tot) else np.nan,
        })
    return pd.DataFrame(rows)


print('=== Time window ===')
sens_time = sweep('window_hours', WINDOW_WIDTHS_TO_TEST,
                  lambda w: select_primary_matchups(matchup_df, w, MIN_VALID_FRACTION))
print(sens_time.to_string(index=False,
      float_format=lambda v: f'{v:.3g}' if pd.notna(v) else 'nan'))

print('\n=== Minimum valid pixel fraction ===')
sens_valid = sweep('min_valid_fraction', VALID_FRACTIONS_TO_TEST,
                   lambda f: select_primary_matchups(matchup_df, TIME_WINDOW_HOURS, f))
print(sens_valid.to_string(index=False,
      float_format=lambda v: f'{v:.3g}' if pd.notna(v) else 'nan'))

print('\n=== DE2000 threshold ===')
sens_de = sweep('de_threshold', DE_THRESHOLDS_TO_TEST,
                lambda t: select_primary_matchups(matchup_df, TIME_WINDOW_HOURS,
                                                  MIN_VALID_FRACTION, de_threshold=t))
print(sens_de.to_string(index=False,
      float_format=lambda v: f'{v:.3g}' if pd.notna(v) else 'nan'))

for nm, d in [('time_window', sens_time), ('valid_fraction', sens_valid),
              ('de_threshold', sens_de)]:
    d.to_csv(os.path.join(STATUS_FOLDER, f'cpr_sensitivity_{nm}.csv'), index=False)
print(f'\nSaved three sensitivity tables to {STATUS_FOLDER}')

## Figures

Generates the four figures used in the Results and Appendix:

1. Distribution of box median DEmin2000 under each LUT (Figure 1 in the dissertation).
2. Satellite AEI against in situ AEI, one panel per taxon variant.
3. Map of matchup locations, coloured by whether an anomaly was detected (Figure 2 in the dissertation).
4. Example eRGB and DE2000 image pairs for individual boxes, for illustration in the Appendix.

In [ ]:
import matplotlib.pyplot as plt

FIG_DIR = os.path.join(OUTPUT_FOLDER, 'figures')
os.makedirs(FIG_DIR, exist_ok=True)


def save_fig(fig, name):
    p = os.path.join(FIG_DIR, name)
    fig.savefig(p, dpi=200, bbox_inches='tight')
    print(f'  saved {p}')


# ---------- Figure 1: binned DE2000 distributions ----------
if len(merged):
    fig, ax = plt.subplots(figsize=(7, 4.5))
    bins = [0, 4, 6, 8, 10, 12, 14, 16, 20, 100]
    ax.hist([merged['median_DE_C2'].dropna(), merged['median_DE_Cal'].dropna()],
            bins=bins, label=['Case 2 LUT', 'Case 2 + Calanus LUT'])
    ax.axvline(DE_THRESHOLD, color='k', ls='--', lw=1,
               label=f'anomaly threshold ({DE_THRESHOLD:g})')
    ax.set_xlabel('box median DEmin2000')
    ax.set_ylabel('number of matched boxes')
    ax.set_title('Anomaly magnitude under each bio-optical model')
    ax.legend()
    plt.tight_layout(); save_fig(fig, 'fig1_de2000_distribution.png'); plt.show()


import matplotlib.pyplot as plt
import numpy as np

# Use Times New Roman throughout the figure, at a larger, more legible size
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman']

TITLE_SIZE = 16
LABEL_SIZE = 14
TICK_SIZE = 14
LEGEND_SIZE = 14
SUPTITLE_SIZE = 16

# Display names for each panel title, matching the italicised species
# formatting used throughout the thesis text. Non-species groupings
# (Calanus I-IV, Calanus adults, All adults, Total) are left in plain text,
# since only genus/species binomials take italics.
DISPLAY_NAMES = {
    'C_fin': r'$\it{C.\ finmarchicus}$',
    'C_helg': r'$\it{C.\ helgolandicus}$',
    'C_typicus': r'$\it{C.\ typicus}$',
    'Calanus_I_IV': 'Calanus I\u2013IV',
    'Calanus_adults': 'Calanus adults',
    'All_adults': 'All adults',
    'Total': 'Total AEI',
}

# ---------- Figure 2 (fixed titles, larger Times New Roman text): satellite vs in situ AEI, one panel per variant ----------
if len(merged) >= 5:
    keep = ['C_fin', 'C_helg', 'C_typicus', 'Calanus_I_IV', 'Calanus_adults', 'All_adults', 'Total']
    vs = [(k, AEI_VARIANTS[k]) for k in keep]
    ncol = 2
    nrow = int(np.ceil(len(vs) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(5.5 * ncol, 4.0 * nrow))
    axes = np.atleast_1d(axes).ravel()
    for ax, (short, col) in zip(axes, vs):
        d = merged[['satellite_aei', col]].dropna()
        bp = d[(d['satellite_aei'] > 0) & (d[col] > 0)]
        ax.scatter(d[col], d['satellite_aei'], s=14, alpha=0.45,
                   color='0.6', label='all matchups')
        if len(bp):
            ax.scatter(bp[col], bp['satellite_aei'], s=18, alpha=0.85,
                       color='tab:red', label='both positive')
        ax.set_xscale('symlog', linthresh=1e-3)
        ax.set_yscale('symlog', linthresh=1e-3)
        row = stats_df[stats_df['variant'] == short]
        display_name = DISPLAY_NAMES.get(short, short)
        if len(row):
            rr = row.iloc[0]
            ttl = f'{display_name}\nrho(all) = {rr.get("spearman_r_all", np.nan):.2f}'
            if pd.notna(rr.get('pearson_r_loglog')):
                ttl += f', r(log) = {rr["pearson_r_loglog"]:.2f} (n = {int(rr["n_both_positive"])})'
            ax.set_title(ttl, fontsize=TITLE_SIZE)
        ax.set_xlabel('in situ AEI (ug/m3)', fontsize=LABEL_SIZE)
        ax.set_ylabel('satellite AEI (ug/m3)', fontsize=LABEL_SIZE)
        ax.tick_params(labelsize=TICK_SIZE)
    for ax in axes[len(vs):]:
        ax.axis('off')
    axes[0].legend(fontsize=LEGEND_SIZE)
    fig.suptitle('Satellite-derived AEI against in situ AEI by taxon', y=1.005, fontsize=SUPTITLE_SIZE)
    plt.tight_layout()
    save_fig(fig, 'fig2_satellite_vs_insitu_AEI_fixed.png')
    plt.show()
# ---------- Figure 3 (restyled): matchup map ----------
if len(merged):
    fig, ax = plt.subplots(figsize=(7, 8), subplot_kw={'projection': ccrs.PlateCarree()})

    # Ocean and land colours matched to the study area map
    ax.set_facecolor('#cfe6f5')  # light blue ocean, matches Image 1
    ax.add_feature(cfeature.LAND, facecolor='#e8dcb5', zorder=0)  # tan/beige land
    ax.add_feature(cfeature.COASTLINE, linewidth=0.6, color='black', zorder=1)

    # Extent widened to match the study area map's bounding box
    ax.set_extent([-14, -2, 47.5, 57], crs=ccrs.PlateCarree())

    no = merged[merged['satellite_aei'] <= 0]
    ys = merged[merged['satellite_aei'] > 0]

    ax.scatter(no['roi_lon'], no['roi_lat'], s=18, color='0.5',
               label=f'no anomaly detected (n = {len(no)})',
               transform=ccrs.PlateCarree(), zorder=2)

    if len(ys):
        sc = ax.scatter(ys['roi_lon'], ys['roi_lat'], s=34,
                        c=ys['satellite_aei'], cmap='autumn_r',
                        edgecolor='k', linewidth=0.4,
                        label=f'anomaly detected (n = {len(ys)})',
                        transform=ccrs.PlateCarree(), zorder=3)
        fig.colorbar(sc, ax=ax, label='satellite AEI (ug/m3)', shrink=0.7)

    # Gridlines styled to match Image 1 (light grey, dashed, with labels)
    gl = ax.gridlines(draw_labels=True, linewidth=0.4, alpha=0.5,
                       color='grey', linestyle='--')
    gl.top_labels = False
    gl.right_labels = False

    # Bold, left-aligned title matching Image 1's title treatment
    fig.suptitle(f'CPR matchups, +/-{TIME_WINDOW_HOURS}h window',
                 fontsize=13, fontweight='bold', x=0.02, ha='left', y=0.98)

    ax.legend(fontsize=8, loc='lower left')
    plt.tight_layout()
    save_fig(fig, 'fig3_matchup_map_styled.png')
    plt.show()
               
# ---------- Figure 4: example boxes ----------
def save_example_figure(row, label_slug, title_note):
    with h5py.File(row['filepath'], 'r') as f:
        red   = f['Rrs_560'][:].astype(np.float64)
        green = f['Rrs_490'][:].astype(np.float64)
        blue  = f['Rrs_442'][:].astype(np.float64)
        lat_e = f['latitude'][:]
        lon_e = f['longitude'][:]
        land  = f['land_mask'][:] if 'land_mask' in f else np.zeros(red.shape, bool)

    rgb = calc_rgb(red, green, blue)
    valid = np.all(np.isfinite(rgb), axis=-1)

    de_map = np.full(valid.shape, np.nan)
    if valid.any():
        de_map[valid] = compute_de2000(rgb[valid].astype(np.float64), lut_c2_lab)[0]

    extent = [lon_e.min(), lon_e.max(), lat_e.min(), lat_e.max()]
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
    fig.suptitle(f"roi_row_id {row['roi_row_id']}, {row['scene_date']}  ({title_note})")

    disp = np.nan_to_num(np.clip(rgb, 0, 1), nan=1.0)
    disp[land] = [0.55, 0.55, 0.55]
    axes[0].imshow(disp, extent=extent, origin='lower', aspect='auto')
    axes[0].plot(row['roi_lon'], row['roi_lat'], marker='*', markersize=16,
                 markerfacecolor='white', markeredgecolor='black',
                 markeredgewidth=1.1, linestyle='none', label='CPR sample')
    axes[0].legend(fontsize=7, loc='upper right')
    axes[0].set_title('eRGB composite')
    axes[0].set_xlabel('Longitude'); axes[0].set_ylabel('Latitude')

    im = axes[1].imshow(de_map, extent=extent, origin='lower', aspect='auto',
                        cmap='jet', vmin=0, vmax=15)
    axes[1].set_facecolor('0.75')
    axes[1].set_title('DEmin2000 (Case 2 LUT)')
    axes[1].set_xlabel('Longitude')
    fig.colorbar(im, ax=axes[1], label='DEmin2000')
    plt.tight_layout()
    save_fig(fig, f"fig4_row{row['roi_row_id']}_{row['scene_date']}_{label_slug}.png")
    plt.show()


if len(merged) == 0 or 'pct_anomalous' not in merged.columns:
    print('No matchups available for example figures.')
else:
    has_red = merged.dropna(subset=['median_DE_reduction'])
    examples = [('highest_anomaly', merged.loc[merged['pct_anomalous'].idxmax()],
                 'highest anomalous pixel fraction')]
    if len(has_red):
        examples += [
            ('best_fit_improvement',
             has_red.loc[has_red['median_DE_reduction'].idxmax()],
             'Calanus LUT improved the fit most'),
            ('worst_fit_improvement',
             has_red.loc[has_red['median_DE_reduction'].idxmin()],
             'Calanus LUT did not improve the fit'),
        ]
    for slug, row, note in examples:
        print(f"\n=== {slug}: roi_row_id {row['roi_row_id']}, "
              f"{row['pct_anomalous']:.1f}% anomalous, "
              f"DE reduction {row.get('median_DE_reduction', float('nan')):.2f} ===")
        save_example_figure(row, slug, note)

## Outputs

Summary of the files this notebook writes to disk, for reproducibility:

- `Status/cpr_satellite_matchups.csv` -- one row per CPR point, with its primary satellite matchup and all in situ AEI variants (used to produce Table 3 and Figure 2).
- `Status/cpr_satellite_matchups_all_candidates.csv` -- every candidate box before matchup filtering, kept so the sensitivity checks above can be re-run without repeating the DE2000 step.
- `Status/cpr_satellite_stats_summary.csv` and `cpr_satellite_stats_targeted.csv` -- correlation statistics for the full dataset and the targeted high-abundance subset.
- `Status/cpr_sensitivity_time_window.csv`, `cpr_sensitivity_valid_fraction.csv`, `cpr_sensitivity_de_threshold.csv` -- the three sensitivity sweeps above.
- `Data/LUT/Output/figures/` -- the four figures described above.

In [ ]:
import sys
print("Python:", sys.version)

import numpy, pandas, scipy, h5py, skimage
for pkg in (numpy, pandas, scipy, h5py, skimage):
    print(pkg.__name__, pkg.__version__)